<a href="https://colab.research.google.com/github/kb0417/french-ewe-translation-transcription/blob/main/notebooks/05_nllb_translation_french_ewe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05 — Traduction français-éwé avec NLLB

Dans les notebooks précédents, nous avons construit deux modèles Seq2Seq LSTM :

- français → éwé ;
- éwé → français.

Ces modèles constituent des baselines Deep Learning entraînées from scratch.  
Cependant, les traductions générées restent très limitées.

Dans ce notebook, nous testons une approche plus avancée basée sur NLLB-200, un modèle multilingue pré-entraîné pour la traduction automatique.

L’objectif est de comparer notre baseline LSTM avec un modèle moderne déjà entraîné sur de nombreuses langues, notamment des langues faiblement dotées.

Nous allons tester deux directions :

- français → éwé ;
- éwé → français.

In [1]:
# On installe les bibliothèques nécessaires.
#
# transformers permet d'utiliser le modèle NLLB depuis Hugging Face.
# sentencepiece est nécessaire pour certains tokenizers multilingues.
# sacrebleu servira plus tard à évaluer automatiquement les traductions.

!pip install transformers sentencepiece sacrebleu accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.6 MB/s eta 0:00:00


In [2]:
# On importe les bibliothèques utiles pour charger le modèle,
# faire des traductions et manipuler les données.

import os
import torch
import pandas as pd

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [4]:
# On vérifie si un GPU est disponible.
# NLLB peut fonctionner sur CPU, mais ce sera beaucoup plus lent.

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Appareil utilisé :", device)

if device == "cuda":
    print("GPU :", torch.cuda.get_device_name(0))

Appareil utilisé : cuda
GPU : Tesla T4


In [5]:
# On charge le modèle NLLB-200 distilled 600M.
#
# Ce modèle est pré-entraîné pour la traduction multilingue.
# Il est plus adapté à une langue peu dotée comme l'éwé qu'un modèle LSTM
# entraîné from scratch sur un petit corpus.

model_name = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(model_name)
nllb_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

nllb_model = nllb_model.to(device)

print("Modèle NLLB chargé avec succès.")

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Modèle NLLB chargé avec succès.


In [6]:
# Codes de langues utilisés par NLLB :
#
# Français : fra_Latn
# Éwé      : ewe_Latn
#
# Le suffixe Latn indique que les textes sont écrits avec l'alphabet latin.

FRENCH_CODE = "fra_Latn"
EWE_CODE = "ewe_Latn"

In [7]:
# Cette fonction traduit une phrase avec NLLB.
#
# src_lang indique la langue source.
# tgt_lang indique la langue cible.
#
# Exemple :
# src_lang = "fra_Latn", tgt_lang = "ewe_Latn" pour français → éwé.
# src_lang = "ewe_Latn", tgt_lang = "fra_Latn" pour éwé → français.

def translate_nllb(text, src_lang, tgt_lang, max_length=128):
    tokenizer.src_lang = src_lang

    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_length
    ).to(device)

    translated_tokens = nllb_model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt_lang),
        max_length=max_length,
        num_beams=4
    )

    translation = tokenizer.batch_decode(
        translated_tokens,
        skip_special_tokens=True
    )[0]

    return translation

In [8]:
# On teste d'abord quelques phrases françaises simples.

phrases_fr = [
    "Bonjour",
    "Comment tu vas ?",
    "Je suis malade",
    "Je veux boire de l'eau",
    "Merci beaucoup",
    "Je vais à la maison"
]

for phrase in phrases_fr:
    traduction = translate_nllb(
        phrase,
        src_lang=FRENCH_CODE,
        tgt_lang=EWE_CODE
    )

    print("Français :", phrase)
    print("Éwé      :", traduction)
    print("-" * 80)

Français : Bonjour
Éwé      : Bonjour
--------------------------------------------------------------------------------
Français : Comment tu vas ?
Éwé      : Aleke miewɔ va yi ?
--------------------------------------------------------------------------------
Français : Je suis malade
Éwé      : Nyemele dɔ lém o.
--------------------------------------------------------------------------------
Français : Je veux boire de l'eau
Éwé      : Nyemedi be mano tsi o.
--------------------------------------------------------------------------------
Français : Merci beaucoup
Éwé      : Míeda akpe na wò ŋutɔ.
--------------------------------------------------------------------------------
Français : Je vais à la maison
Éwé      : Meyi aƒe me.
--------------------------------------------------------------------------------


In [9]:
# On monte Google Drive pour charger le fichier test.csv du projet.

from google.colab import drive
drive.mount('/content/drive')

project_dir = "/content/drive/MyDrive/french_ewe_project"
processed_dir = os.path.join(project_dir, "data/processed")

test_df = pd.read_csv(os.path.join(processed_dir, "test.csv"))

test_df.head()

Mounted at /content/drive


,french,ewe,source,type,french_length,ewe_length
0,"En Europe, la valeur limite de consommation de...","le yuropa la, fɔ̃nyinuɖuɖu ɖuɖu se ɖe miligraŋ...",https://l-frii.com/,News,15,13
1,Le calao qui ne sait ni nager ni pêcher ne tro...,Kalao yi matem aƒu tsi alo aɖe lã o la mega ny...,https://carnet-de-contes.jimdofree.com/contes/...,Tales,15,15
2,"Ensuite, elle avait rejoint les Etats Unis pou...",emegbea eva Amerika be yeasrɔ̃ tso gaŋutinyawo...,https://leschroniquesdetchonte.com/,Blog,13,8
3,"Dans €œSoufi, mon amour€_x009d_, j'en ai appri...","le «soufi mon amour» me la, mesrɔ̃nu kuɖe mɔsl...",https://leschroniquesdetchonte.com/,Blog,19,13
4,les parents se doivent d'être exemplaires et d...,to esiawo mee miate ŋu atrɔ ɖevi ade ƒe nɔnɔme,https://ellecitoyenne.com/,"Blog, News",19,10


In [10]:
# On teste quelques phrases éwé issues du jeu de test.
# Pour chaque phrase, on affiche :
# - la phrase éwé ;
# - la traduction française attendue ;
# - la traduction générée par NLLB.

for i in range(5):
    ewe_sentence = test_df.iloc[i]["ewe"]
    true_french = test_df.iloc[i]["french"]

    predicted_french = translate_nllb(
        ewe_sentence,
        src_lang=EWE_CODE,
        tgt_lang=FRENCH_CODE
    )

    print("Éwé original       :", ewe_sentence)
    print("Français attendu   :", true_french)
    print("Français NLLB      :", predicted_french)
    print("-" * 100)

Éwé original       : le yuropa la, fɔ̃nyinuɖuɖu ɖuɖu se ɖe miligraŋu 100 le gbe ɖeka kol
Français attendu   : En Europe, la valeur limite de consommation de l’acide glycyrrhizique est limitée à 100 mg/jour
Français NLLB      : En Europe, il est recommandé de consommer moins de 100 milligrammes par jour.
----------------------------------------------------------------------------------------------------
Éwé original       : Kalao yi matem aƒu tsi alo aɖe lã o la mega nya nusi woaɖu o.
Français attendu   : Le calao qui ne sait ni nager ni pêcher ne trouva donc plus à manger.
Français NLLB      : On ne mange pas d'eau, on ne mange pas de viande et on ne sait pas quoi manger.
----------------------------------------------------------------------------------------------------
Éwé original       : emegbea eva Amerika be yeasrɔ̃ tso gaŋutinyawo ŋui
Français attendu   : Ensuite, elle avait rejoint les Etats Unis pour préparer un diplôme en finances
Français NLLB      : L'Amérique a besoin d'ap

In [11]:
# On teste maintenant quelques phrases françaises du jeu de test.
# Cela permet de comparer NLLB avec les traductions éwé attendues du dataset.

for i in range(5):
    french_sentence = test_df.iloc[i]["french"]
    true_ewe = test_df.iloc[i]["ewe"]

    predicted_ewe = translate_nllb(
        french_sentence,
        src_lang=FRENCH_CODE,
        tgt_lang=EWE_CODE
    )

    print("Français original :", french_sentence)
    print("Éwé attendu       :", true_ewe)
    print("Éwé NLLB          :", predicted_ewe)
    print("-" * 100)

Français original : En Europe, la valeur limite de consommation de l’acide glycyrrhizique est limitée à 100 mg/jour
Éwé attendu       : le yuropa la, fɔ̃nyinuɖuɖu ɖuɖu se ɖe miligraŋu 100 le gbe ɖeka kol
Éwé NLLB          : Le Europa la, wodzra glycyrrhizic acid ƒe dzidzenu si nye 100 mg/ŋkeke dzi.
----------------------------------------------------------------------------------------------------
Français original : Le calao qui ne sait ni nager ni pêcher ne trouva donc plus à manger.
Éwé attendu       : Kalao yi matem aƒu tsi alo aɖe lã o la mega nya nusi woaɖu o.
Éwé NLLB          : Le klao si menya tsiƒu alo tɔƒodela o ta la, mekpɔa nu si wòɖuna o.
----------------------------------------------------------------------------------------------------
Français original : Ensuite, elle avait rejoint les Etats Unis pour préparer un diplôme en finances
Éwé attendu       : emegbea eva Amerika be yeasrɔ̃ tso gaŋutinyawo ŋui
Éwé NLLB          : Emegbe eva ɖo United States be yeawɔ sukudede l

In [12]:
# On prend un petit échantillon du jeu de test pour évaluer rapidement NLLB.
# L'objectif est d'obtenir une première idée de la qualité des traductions.

sample_size = 50
sample_df = test_df.sample(sample_size, random_state=42).reset_index(drop=True)

In [13]:
# Traductions NLLB éwé → français sur l'échantillon.

predictions_ewe_to_fr = []
references_fr = []

for i in range(len(sample_df)):
    ewe_sentence = sample_df.iloc[i]["ewe"]
    true_french = sample_df.iloc[i]["french"]

    predicted = translate_nllb(
        ewe_sentence,
        src_lang=EWE_CODE,
        tgt_lang=FRENCH_CODE
    )

    predictions_ewe_to_fr.append(predicted)
    references_fr.append(true_french)

    if (i + 1) % 10 == 0:
        print(f"{i + 1}/{len(sample_df)} phrases traduites")

10/50 phrases traduites
20/50 phrases traduites
30/50 phrases traduites
40/50 phrases traduites
50/50 phrases traduites


In [14]:
# On calcule le score BLEU.
# BLEU compare les traductions générées aux traductions de référence.
# Ce n'est pas parfait, mais c'est une métrique classique en traduction automatique.

import sacrebleu

bleu_ewe_to_fr = sacrebleu.corpus_bleu(
    predictions_ewe_to_fr,
    [references_fr]
)

print("BLEU éwé → français :", bleu_ewe_to_fr.score)

BLEU éwé → français : 2.766330620374854


In [15]:
# Traductions NLLB français → éwé sur le même échantillon.

predictions_fr_to_ewe = []
references_ewe = []

for i in range(len(sample_df)):
    french_sentence = sample_df.iloc[i]["french"]
    true_ewe = sample_df.iloc[i]["ewe"]

    predicted = translate_nllb(
        french_sentence,
        src_lang=FRENCH_CODE,
        tgt_lang=EWE_CODE
    )

    predictions_fr_to_ewe.append(predicted)
    references_ewe.append(true_ewe)

    if (i + 1) % 10 == 0:
        print(f"{i + 1}/{len(sample_df)} phrases traduites")

10/50 phrases traduites
20/50 phrases traduites
30/50 phrases traduites
40/50 phrases traduites
50/50 phrases traduites


In [16]:
bleu_fr_to_ewe = sacrebleu.corpus_bleu(
    predictions_fr_to_ewe,
    [references_ewe]
)

print("BLEU français → éwé :", bleu_fr_to_ewe.score)

BLEU français → éwé : 3.5779131358627656


In [17]:
# On crée un tableau pour comparer les performances déjà obtenues.
#
# Les modèles LSTM ont été évalués avec l'accuracy mot par mot.
# NLLB est évalué ici avec BLEU sur un échantillon.
# Les métriques ne sont donc pas strictement identiques,
# mais elles donnent une première comparaison qualitative.

results = pd.DataFrame({
    "Modèle": [
        "Seq2Seq LSTM",
        "Seq2Seq LSTM",
        "NLLB pré-entraîné",
        "NLLB pré-entraîné"
    ],
    "Direction": [
        "Français → Éwé",
        "Éwé → Français",
        "Français → Éwé",
        "Éwé → Français"
    ],
    "Métrique": [
        "Accuracy mot par mot",
        "Accuracy mot par mot",
        "BLEU sur 50 phrases",
        "BLEU sur 50 phrases"
    ],
    "Score": [
        0.2576,
        0.2400,
        bleu_fr_to_ewe.score,
        bleu_ewe_to_fr.score
    ]
})

results

,Modèle,Direction,Métrique,Score
0,Seq2Seq LSTM,Français → Éwé,Accuracy mot par mot,0.257600
1,Seq2Seq LSTM,Éwé → Français,Accuracy mot par mot,0.240000
2,NLLB pré-entraîné,Français → Éwé,BLEU sur 50 phrases,3.577913
3,NLLB pré-entraîné,Éwé → Français,BLEU sur 50 phrases,2.766331


Les métriques ne sont pas directement comparables : les modèles LSTM sont évalués avec une accuracy mot par mot, tandis que NLLB est évalué avec BLEU sur un échantillon. La comparaison sert donc surtout à compléter l’analyse qualitative.

In [18]:
# On sauvegarde le tableau de comparaison dans Drive.
# Il pourra être utilisé dans le rapport final.

results_path = os.path.join(project_dir, "reports", "comparison_lstm_nllb.csv")

os.makedirs(os.path.dirname(results_path), exist_ok=True)

results.to_csv(results_path, index=False)

print("Résultats sauvegardés ici :", results_path)

Résultats sauvegardés ici : /content/drive/MyDrive/french_ewe_project/reports/comparison_lstm_nllb.csv


## Conclusion

Dans ce notebook, nous avons testé le modèle NLLB-200 distilled 600M pour la traduction français-éwé et éwé-français.

Contrairement aux modèles Seq2Seq LSTM entraînés from scratch, NLLB est un modèle multilingue pré-entraîné. Il dispose donc déjà de connaissances linguistiques acquises sur de grands corpus multilingues.

Les premiers tests qualitatifs montrent que NLLB produit des traductions beaucoup plus cohérentes que les modèles LSTM. Par exemple, des phrases simples comme “Je suis malade”, “Merci beaucoup” ou “Je vais à la maison” sont traduites de manière plus naturelle en éwé.

L’évaluation automatique avec BLEU sur un échantillon de 50 phrases donne les résultats suivants :

- BLEU français → éwé : 3.58
- BLEU éwé → français : 2.77

Ces scores restent faibles, mais ils doivent être interprétés avec prudence. BLEU pénalise fortement les traductions qui ne correspondent pas exactement à la référence, même lorsque le sens général est proche.

Cette étape confirme que les modèles pré-entraînés sont plus adaptés à la traduction automatique pour une langue peu dotée comme l’éwé. Les modèles LSTM restent utiles comme baselines pédagogiques, tandis que NLLB constitue une base plus solide pour obtenir un système de traduction réellement exploitable.

La prochaine étape logique consistera à adapter NLLB à notre corpus parallèle français-éwé grâce à un fine-tuning.